In [ ]:
# Cellule 1 : Paramètres de connexion
server = 'localhost'  # ou 'DESKTOP-3U42S6P'
database = 'event_DWH'

In [ ]:
# Cellule 2 : Essayer de se connecter sans identifiants (Windows Authentication)
from sqlalchemy import create_engine

# Version Windows Authentication (utilise ton utilisateur Windows)
connection_string = f"mssql+pyodbc://@{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"

print("Tentative de connexion...")
print(connection_string)

try:
    engine = create_engine(connection_string)
    connection = engine.connect()
    print("✅ CONNEXION RÉUSSIE !")
    connection.close()
except Exception as e:
    print(f"❌ Erreur: {e}")

In [ ]:
# Cellule 3 : Tester la lecture d'une table
import pandas as pd

# Test sur une petite table
df_test = pd.read_sql("SELECT TOP 5 * FROM Dim_Beneficiary", engine)

print("✅ Données chargées !")
print(f"Shape: {df_test.shape}")
print("\nAperçu:")
print(df_test.head())

In [ ]:
# Cellule 4 : Compter les lignes
count_query = "SELECT COUNT(*) as total FROM FACT_VENTES"
total = pd.read_sql(count_query, engine)

print(f"📊 Nombre total de réservations: {total['total'][0]:,} lignes")

In [ ]:
# Cellule : Chargement complet CORRIGÉ
print("⏳ Chargement des données...")

query = """
SELECT 
    f.id_reservation,
    f.price,
    f.nbr_reservations,
    f.nbr_visitors,
    f.marketing_spend,
    f.market_count,
    f.status as reservation_status,
    e.title as event_title,
    e.type as event_type,
    e.event_date,
    cat.name as category_name,
    l.city,
    l.country,
    ev.rating,
    ev.comment,
    t.trend_score,
    t.growth_rate_pct,
    v.capacity_min,
    v.capacity_max,
    v.venue_type,
    ent.name as entertainer_name,
    ent.categorie as entertainer_category,
    ent.nationality,
    ent.followers_instagram,
    ent.followers_tiktok,
    ent.average_price as entertainer_price
FROM FACT_VENTES f
LEFT JOIN Dim_Event e ON f.id_event = e.id_event
LEFT JOIN Dim_Category cat ON f.id_category = cat.id_category
LEFT JOIN Dim_Localisation l ON f.id_localisation = l.id_localisation
LEFT JOIN Dim_Evaluation ev ON f.id_evaluation = ev.id_evaluation
LEFT JOIN Dim_Trends t ON f.id_trend = t.id_trend
LEFT JOIN Dim_Venue v ON f.id_venue = v.id_venue
LEFT JOIN Dim_Entertainer ent ON f.id_entertainer = ent.id_entertainer
"""

df = pd.read_sql(query, engine)

print(f"✅ Chargé: {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"\n📋 Colonnes disponibles:")
print(df.columns.tolist())


# Sauvegarde en CSV
df.to_csv('event_data.csv', index=False)
print("✅ Données sauvegardées dans 'event_data.csv'")

# Aperçu
print("\n📊 Aperçu:")
print(df.head())

In [ ]:
# Cellule : Aperçu des données
print("📊 Aperçu:")
print(df.head())

print("\n🔢 Types de données:")
print(df.dtypes)

print("\n📊 Stats rapides:")
print(df.describe())

In [ ]:
# Cellule : Info générale
print(f"📊 Taille: {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"\n📋 Liste des colonnes:")
for i, col in enumerate(df.columns):
    print(f"   {i+1}. {col}")

In [ ]:
# Cellule : Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Manquantes': missing, 'Pourcentage': missing_pct})
missing_df = missing_df[missing_df['Manquantes'] > 0].sort_values('Pourcentage', ascending=False)

print("🔍 Valeurs manquantes:")
if len(missing_df) > 0:
    print(missing_df)
else:
    print("✅ Aucune valeur manquante !")

In [ ]:
# Cellule : Identifier et retirer les colonnes ID
# Colonnes à exclure de l'analyse statistique
id_columns = ['id_reservation', 'id_event', 'id_category', 'id_localisation', 
              'id_evaluation', 'id_trend', 'id_venue', 'id_entertainer']

# Vérifier lesquelles existent vraiment
id_columns_exist = [col for col in id_columns if col in df.columns]

print(f"🔍 Colonnes ID à exclure: {id_columns_exist}")

# Créer un DataFrame sans les ID pour l'analyse
df_clean = df.drop(columns=id_columns_exist)

print(f"\n✅ DataFrame nettoyé: {df_clean.shape[0]} lignes, {df_clean.shape[1]} colonnes")
print(f"   (suppression de {len(id_columns_exist)} colonnes ID)")

In [ ]:
# Cellule : Statistiques sur les colonnes numériques (sans les ID)
print("📊 Statistiques des colonnes numériques (sans ID):")
numeric_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns
print(f"Colonnes numériques analysées: {numeric_cols.tolist()}")
print("\n")
df_clean[numeric_cols].describe()

In [ ]:
# Cellule : Voir les colonnes restantes
print("📋 Toutes les colonnes après nettoyage:")
for i, col in enumerate(df_clean.columns):
    print(f"   {i+1}. {col} ({df_clean[col].dtype})")

In [ ]:
# Cellule : Valeurs manquantes sur le DataFrame nettoyé
missing = df_clean.isnull().sum()
missing_pct = (missing / len(df_clean)) * 100
missing_df = pd.DataFrame({'Manquantes': missing, 'Pourcentage': missing_pct})
missing_df = missing_df[missing_df['Manquantes'] > 0].sort_values('Pourcentage', ascending=False)

print("🔍 Valeurs manquantes après nettoyage:")
if len(missing_df) > 0:
    print(missing_df)
else:
    print("✅ Aucune valeur manquante !")

In [ ]:
# Cellule : Distribution des réservations
print("📈 Distribution du nombre de réservations:")
print(df['nbr_reservations'].describe())

# Créer la target (client fidèle = plus de 2 réservations)
df['target_fidele'] = (df['nbr_reservations'] > 2).astype(int)
print(f"\n🎯 Clients fidèles (target=1): {df['target_fidele'].sum():.0f}")
print(f"   Clients non fidèles (target=0): {(df['target_fidele'] == 0).sum():.0f}")
print(f"   Taux de fidélité: {df['target_fidele'].mean()*100:.1f}%")

In [ ]:
# ============================================
# CLASSIFICATION AVEC PIPELINE + GRIDSEARCH (CORRIGÉE)
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score, f1_score, accuracy_score

# Modèles
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

print("="*60)
print("📊 CLASSIFICATION - AVEC PIPELINE + GRIDSEARCH")
print("="*60)

# ============================================
# 1. CHARGEMENT DES DONNÉES
# ============================================
df = pd.read_csv('event_data.csv')

# 🔧 CORRECTION: Target basée sur la médiane (pas sur >=3)
median_reservations = df['nbr_reservations'].median()
df['target'] = (df['nbr_reservations'] > median_reservations).astype(int)

print(f"📊 Médiane des réservations: {median_reservations}")
print(f"🎯 Distribution target:")
print(f"   Classe 1 (fidèle): {df['target'].sum()} ({df['target'].mean()*100:.1f}%)")
print(f"   Classe 0 (non fidèle): {(df['target']==0).sum()} ({(1-df['target'].mean())*100:.1f}%)")

# Vérifier qu'on a bien 2 classes
if df['target'].nunique() < 2:
    raise ValueError("❌ Impossible de faire de la classification: une seule classe détectée!")

# Features
feature_cols = ['price', 'nbr_visitors', 'marketing_spend', 'market_count', 
                'trend_score', 'rating', 'followers_instagram', 'average_price']
feature_cols = [col for col in feature_cols if col in df.columns]

X = df[feature_cols].copy()
y = df['target'].copy()

# Gestion des NaN
for col in X.columns:
    if X[col].isnull().sum() > 0:
        X[col].fillna(X[col].median(), inplace=True)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 Train: {X_train.shape}, Test: {X_test.shape}")
print(f"🎯 Target train: {y_train.mean()*100:.1f}% de fidèles")

# ============================================
# 2. PIPELINE + GRIDSEARCH POUR CHAQUE MODÈLE
# ============================================

results = {}

# -------------------------------------------------
# MODÈLE 1: Decision Tree
# -------------------------------------------------
print("\n" + "="*50)
print("🌳 1. DECISION TREE")
print("="*50)

pipeline_dt = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

param_grid_dt = {
    'classifier__max_depth': [3, 5, 10],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2]
}

grid_dt = GridSearchCV(pipeline_dt, param_grid_dt, cv=5, scoring='f1', n_jobs=-1)
grid_dt.fit(X_train, y_train)

print(f"✅ Meilleurs paramètres: {grid_dt.best_params_}")
print(f"✅ Meilleur F1 (CV): {grid_dt.best_score_:.4f}")

y_pred_dt = grid_dt.predict(X_test)

# Vérifier que predict_proba fonctionne (2 classes)
if len(grid_dt.classes_) == 2:
    y_proba_dt = grid_dt.predict_proba(X_test)[:, 1]
    auc_dt = roc_auc_score(y_test, y_proba_dt)
    print(f"📊 ROC-AUC: {auc_dt:.4f}")
else:
    auc_dt = 0.5
    print("⚠️ Une seule classe détectée, ROC-AUC non applicable")

f1_dt = f1_score(y_test, y_pred_dt)
print(f"📊 Test - Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}, F1: {f1_dt:.4f}")

results['Decision Tree'] = {'f1': f1_dt, 'auc': auc_dt, 'model': grid_dt, 'y_pred': y_pred_dt}

# -------------------------------------------------
# MODÈLE 2: Random Forest
# -------------------------------------------------
print("\n" + "="*50)
print("🌲 2. RANDOM FOREST")
print("="*50)

pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10],
    'classifier__min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(pipeline_rf, param_grid_rf, cv=5, scoring='f1', n_jobs=-1)
grid_rf.fit(X_train, y_train)

print(f"✅ Meilleurs paramètres: {grid_rf.best_params_}")
print(f"✅ Meilleur F1 (CV): {grid_rf.best_score_:.4f}")

y_pred_rf = grid_rf.predict(X_test)

if len(grid_rf.classes_) == 2:
    y_proba_rf = grid_rf.predict_proba(X_test)[:, 1]
    auc_rf = roc_auc_score(y_test, y_proba_rf)
    print(f"📊 ROC-AUC: {auc_rf:.4f}")
else:
    auc_rf = 0.5

f1_rf = f1_score(y_test, y_pred_rf)
print(f"📊 Test - Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}, F1: {f1_rf:.4f}")

results['Random Forest'] = {'f1': f1_rf, 'auc': auc_rf, 'model': grid_rf, 'y_pred': y_pred_rf}

# -------------------------------------------------
# MODÈLE 3: XGBoost
# -------------------------------------------------
print("\n" + "="*50)
print("🚀 3. XGBOOST")
print("="*50)

pipeline_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
])

param_grid_xgb = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [3, 5],
    'classifier__learning_rate': [0.01, 0.1]
}

grid_xgb = GridSearchCV(pipeline_xgb, param_grid_xgb, cv=5, scoring='f1', n_jobs=-1)
grid_xgb.fit(X_train, y_train)

print(f"✅ Meilleurs paramètres: {grid_xgb.best_params_}")
print(f"✅ Meilleur F1 (CV): {grid_xgb.best_score_:.4f}")

y_pred_xgb = grid_xgb.predict(X_test)

if len(grid_xgb.classes_) == 2:
    y_proba_xgb = grid_xgb.predict_proba(X_test)[:, 1]
    auc_xgb = roc_auc_score(y_test, y_proba_xgb)
    print(f"📊 ROC-AUC: {auc_xgb:.4f}")
else:
    auc_xgb = 0.5

f1_xgb = f1_score(y_test, y_pred_xgb)
print(f"📊 Test - Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}, F1: {f1_xgb:.4f}")

results['XGBoost'] = {'f1': f1_xgb, 'auc': auc_xgb, 'model': grid_xgb, 'y_pred': y_pred_xgb}

# -------------------------------------------------
# MODÈLE 4: SVM
# -------------------------------------------------
print("\n" + "="*50)
print("⚡ 4. SVM")
print("="*50)

pipeline_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SVC(random_state=42, probability=True))
])

param_grid_svm = {
    'classifier__C': [0.1, 1, 10],
    'classifier__kernel': ['rbf'],
    'classifier__gamma': ['scale']
}

grid_svm = GridSearchCV(pipeline_svm, param_grid_svm, cv=3, scoring='f1', n_jobs=-1)
grid_svm.fit(X_train, y_train)

print(f"✅ Meilleurs paramètres: {grid_svm.best_params_}")
print(f"✅ Meilleur F1 (CV): {grid_svm.best_score_:.4f}")

y_pred_svm = grid_svm.predict(X_test)

if len(grid_svm.classes_) == 2:
    y_proba_svm = grid_svm.predict_proba(X_test)[:, 1]
    auc_svm = roc_auc_score(y_test, y_proba_svm)
    print(f"📊 ROC-AUC: {auc_svm:.4f}")
else:
    auc_svm = 0.5

f1_svm = f1_score(y_test, y_pred_svm)
print(f"📊 Test - Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}, F1: {f1_svm:.4f}")

results['SVM'] = {'f1': f1_svm, 'auc': auc_svm, 'model': grid_svm, 'y_pred': y_pred_svm}

# ============================================
# 3. COMPARAISON FINALE
# ============================================
print("\n" + "="*60)
print("📊 COMPARAISON DES 4 MODÈLES")
print("="*60)

comparison = pd.DataFrame([
    {'Modèle': name, 'F1-Score': res['f1'], 'ROC-AUC': res['auc']}
    for name, res in results.items()
])
print(comparison.to_string(index=False))

best_model = comparison.loc[comparison['F1-Score'].idxmax(), 'Modèle']
print(f"\n🏆 MEILLEUR MODÈLE: {best_model}")

# ============================================
# 4. VISUALISATIONS
# ============================================

# 4.1 Matrices de confusion
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
models_list = list(results.keys())

for i, (name, res) in enumerate(results.items()):
    ax = axes[i//2, i%2]
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
    ax.set_title(f'{name} - Matrice de confusion')
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel')

plt.tight_layout()
plt.savefig('classification_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

# 4.2 Courbes ROC (uniquement si AUC valide)
plt.figure(figsize=(10, 8))
colors = {'Decision Tree': 'green', 'Random Forest': 'blue', 'XGBoost': 'orange', 'SVM': 'red'}

for name, res in results.items():
    if res['auc'] > 0.5:  # Seulement si la classification a fonctionné
        y_proba = res['model'].predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        plt.plot(fpr, tpr, label=f"{name} (AUC = {res['auc']:.3f})", color=colors[name], linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Aléatoire')
plt.xlabel('Taux de faux positifs')
plt.ylabel('Taux de vrais positifs')
plt.title('Courbes ROC - Comparaison des modèles')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('classification_roc_pipeline.png', dpi=300, bbox_inches='tight')
plt.show()

# 4.3 Graphique comparatif
plt.figure(figsize=(10, 6))
comparison_melted = comparison.melt(id_vars='Modèle', var_name='Métrique', value_name='Score')
sns.barplot(data=comparison_melted, x='Métrique', y='Score', hue='Modèle')
plt.ylim(0, 1)
plt.title('Comparaison des modèles - F1 et ROC-AUC')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('classification_comparison_pipeline.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("✅ CLASSIFICATION TERMINÉE - Critères validés !")
print("="*60)

In [ ]:
# Cellule : Version CORRIGÉE - avec tous les imports
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

print("="*60)
print("🔧 CLASSIFICATION - VERSION CORRIGÉE (sans leakage)")
print("="*60)

# Features PLUS SÛRES - on enlève celles qui pourraient être liées à la target
feature_cols_safe = [
    'price',           # Prix du service
    'marketing_spend', # Budget marketing  
    'market_count',    # Niveau de compétition
    'trend_score',     # Score de tendance
    'rating',          # Note
    'average_price'    # Prix moyen de l'artiste
]

# Garder celles qui existent
feature_cols_safe = [col for col in feature_cols_safe if col in df.columns]

X_safe = df[feature_cols_safe].copy()
y = df['target'].copy()

# Gestion des NaN
for col in X_safe.columns:
    if X_safe[col].isnull().sum() > 0:
        X_safe[col].fillna(X_safe[col].median(), inplace=True)

print(f"📋 Features utilisées: {feature_cols_safe}")

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_safe, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📊 Train: {X_train.shape}, Test: {X_test.shape}")
print(f"🎯 Target train: {y_train.mean()*100:.1f}% de fidèles")

# ============================================
# RANDOM FOREST (seul modèle fiable)
# ============================================
print("\n" + "="*50)
print("🌲 RANDOM FOREST (version corrigée)")
print("="*50)

pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [5, 10],
    'classifier__min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(pipeline_rf, param_grid_rf, cv=5, scoring='f1', n_jobs=-1)
grid_rf.fit(X_train, y_train)

print(f"✅ Meilleurs paramètres: {grid_rf.best_params_}")
print(f"✅ Meilleur F1 (CV): {grid_rf.best_score_:.4f}")

y_pred_rf = grid_rf.predict(X_test)
y_proba_rf = grid_rf.predict_proba(X_test)[:, 1]

print(f"\n📊 Performance sur le TEST:")
print(f"   Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"   Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"   Recall: {recall_score(y_test, y_pred_rf):.4f}")
print(f"   F1-Score: {f1_score(y_test, y_pred_rf):.4f}")
print(f"   ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}")

# Matrice de confusion
plt.figure(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de confusion - Random Forest (corrigé)')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.show()

# Feature importance
plt.figure(figsize=(8, 5))
importance_df = pd.DataFrame({
    'feature': feature_cols_safe,
    'importance': grid_rf.best_estimator_.named_steps['classifier'].feature_importances_
}).sort_values('importance', ascending=True)

plt.barh(importance_df['feature'], importance_df['importance'], color='steelblue')
plt.xlabel('Importance')
plt.title('Importance des features - Random Forest')
plt.tight_layout()
plt.show()